In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
from torchvision.transforms.v2.functional import to_pil_image

%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import torch.nn.functional as F
from common import *

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors_darker = sns.color_palette("dark", 5 * 2)
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
import os


def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
ratios = [0.05, 0.1, 0.2, 0.5]
lambdas = 0.005
loss = []
acc_object = []
acc_background = []
steps = []
eig_list = []
method = "waterbirds_c"
directory = f"datas/lambda_0.005_simple_color"
print(directory)
for r in ratios:
    loss_array = load_from_csv(f"{directory}/{r}/loss.csv")
    [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{r}/accuracy.csv")
    eigvals_lists_array = load_from_csv_split(f"{directory}/{r}/eig.csv")
    loss.append(loss_array)
    acc_object.append(acc_object_array)
    acc_background.append(acc_background_array)
    steps.append(step_array)
    eig_list.append(eigvals_lists_array)

In [ ]:
background_predictability = 70

fig, axs = plt.subplots(1, 1, figsize=(13, 5.3))

positions = []
plateau_range = 3
for r in range(len(ratios)):
    pos = np.where(acc_object[r] > background_predictability + plateau_range)
    if len(pos[0]) > 0:
        positions.append(pos[0][0])
        if r != 0:
            axs.axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
            axs.axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
    axs.plot(steps[r][:140], acc_object[r][:140], alpha=0.6, linewidth=5, color=colors_darker[r],
             label=f"Object ratio : {ratios[r]:.3f}")

upper_y = np.zeros_like(steps[0]) + background_predictability + plateau_range
lower_y = np.zeros_like(steps[0]) + background_predictability - plateau_range
axs.fill_between(steps[0], upper_y, lower_y, color='skyblue', alpha=0.4)
axs.axhline(background_predictability, linestyle="--", color="black", linewidth=2, alpha=0.4)
axs.set_xlabel("Step")
axs.set_ylabel("Linear Probing Accuracy")
margin_x = 1400 * 0.02
margin_y = 20 * 0.1
axs.set_xlim(0 - margin_x, 1400 + margin_x)
axs.set_ylim(65 - margin_y, 80 + margin_y)
axs.text(520, 68.5, "background-label correlation")

axs.legend(ncols=2, loc='lower right')

plt.tight_layout()
plt.savefig(f'figure/{method}.pdf', format='pdf')
plt.show()